# Facebook MMS-TTS for Vietnamese

## What is MMS-TTS?

**Massively Multilingual Speech (MMS)** is a project by Meta (Facebook) AI Research that brings speech technology to over 1,100 languages. The TTS component uses the **VITS** (Variational Inference with adversarial learning for end-to-end Text-to-Speech) architecture to generate speech from text.

### Key facts

- **Model**: `facebook/mms-tts-vie` on HuggingFace
- **Architecture**: VITS (Variational Inference TTS) -- a conditional variational autoencoder with adversarial training
- **Coverage**: 1,100+ languages, including Vietnamese
- **Speaker**: Single speaker per language (no voice cloning)
- **Hardware**: Lightweight enough to run on CPU -- no GPU required
- **License**: CC BY-NC 4.0

### How VITS works (simplified)

VITS combines three components into a single end-to-end model:
1. **Text encoder** -- converts input text (phonemes) into hidden representations
2. **Stochastic duration predictor** -- determines the length of each phoneme
3. **Decoder (HiFi-GAN)** -- generates the raw audio waveform

Unlike two-stage systems (text-to-mel + vocoder), VITS produces audio directly from text in one pass.

## 1. Installation

We need `transformers` for the model, `scipy` for saving WAV files, and `torch` as the backend.

In [ ]:
!pip install -q transformers scipy torch

## 2. Load Model and Tokenizer

The MMS-TTS Vietnamese model uses a character-level tokenizer that maps Vietnamese text (including diacritics and tones) into token IDs. The model then converts these tokens into audio.

On first run, the model (~300 MB) will be downloaded from HuggingFace.

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch
import scipy.io.wavfile

model = VitsModel.from_pretrained("facebook/mms-tts-vie")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-vie")

print(f"Model loaded successfully.")
print(f"Sampling rate: {model.config.sampling_rate} Hz")

## 3. Basic Speech Generation

The workflow is straightforward:
1. Tokenize the Vietnamese text
2. Pass tokens through the model to generate a waveform
3. Save as WAV and play back

Let's start with a simple greeting.

In [ ]:
from IPython.display import Audio, display

def speak(text, filename=None):
    """Generate speech from Vietnamese text using MMS-TTS."""
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        output = model(**inputs)

    waveform = output.waveform[0].cpu().numpy()
    sample_rate = model.config.sampling_rate

    if filename:
        scipy.io.wavfile.write(filename, rate=sample_rate, data=waveform)
        print(f"Saved: {filename}")

    display(Audio(waveform, rate=sample_rate, autoplay=False))
    return waveform

In [ ]:
text = "Xin chào, tôi là trợ lý ảo."
print(f"Text: {text}")
waveform = speak(text, "output_basic.wav")

## 4. Multiple Examples -- Vietnamese Tones and Sentence Types

Vietnamese is a tonal language with six tones. Let's test several sentences covering different tones, lengths, and contexts to hear how the model handles them.

In [ ]:
examples = [
    ("Hôm nay thời tiết rất đẹp.", "example_weather.wav"),
    ("Việt Nam là một đất nước xinh đẹp với nhiều di sản văn hóa.", "example_culture.wav"),
    ("Bạn có khỏe không? Tôi rất vui được gặp bạn.", "example_greeting.wav"),
    ("Trí tuệ nhân tạo đang thay đổi thế giới mỗi ngày.", "example_ai.wav"),
    ("Cảm ơn bạn đã lắng nghe.", "example_thanks.wav"),
]

for text, filename in examples:
    print(f"\n--- Text: {text}")
    speak(text, filename)

## 5. Long Text Generation

Let's try a longer paragraph about Vietnamese culture. VITS models can handle longer inputs, but very long texts may see quality degradation. For production use, splitting into sentences is recommended.

In [ ]:
long_text = (
    "Việt Nam có một nền văn hóa phong phú và đa dạng, "
    "được hình thành qua hàng nghìn năm lịch sử. "
    "Từ những làn điệu dân ca quan họ Bắc Ninh đến nghệ thuật ca trù, "
    "âm nhạc truyền thống Việt Nam mang đậm bản sắc dân tộc. "
    "Ẩm thực Việt Nam nổi tiếng thế giới với phở, bánh mì và bún chả. "
    "Mỗi vùng miền đều có những món ăn đặc trưng riêng, "
    "tạo nên sự đa dạng trong văn hóa ẩm thực của đất nước."
)

print(f"Text length: {len(long_text)} characters")
print(f"Text: {long_text}\n")
waveform = speak(long_text, "output_long.wav")
print(f"Audio duration: {len(waveform) / model.config.sampling_rate:.2f} seconds")

## 6. Model Inspection

Let's look under the hood at the model architecture and size.

In [ ]:
# Model size and parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
model_size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2)

print("=" * 50)
print("MODEL INFORMATION")
print("=" * 50)
print(f"Model name:       facebook/mms-tts-vie")
print(f"Architecture:     VITS (Variational Inference TTS)")
print(f"Total parameters: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Model size:       {model_size_mb:.1f} MB")
print(f"Sampling rate:    {model.config.sampling_rate} Hz")
print(f"Vocab size:       {model.config.vocab_size}")

In [ ]:
# Architecture overview -- main components
print("\nARCHITECTURE COMPONENTS")
print("=" * 50)
for name, module in model.named_children():
    num_params = sum(p.numel() for p in module.parameters())
    print(f"  {name:30s} -> {num_params:>10,} params")

### Architecture breakdown

| Component | Role |
|-----------|------|
| **text_encoder** | Converts tokenized text into latent representations using transformer layers |
| **duration_predictor** | Predicts how long each phoneme/character should last |
| **flow** | Normalizing flow that transforms simple distributions into complex speech patterns |
| **decoder** | HiFi-GAN vocoder that generates the final audio waveform |

## 7. Limitations

It is important to understand what MMS-TTS can and cannot do:

**Strengths:**
- Fully offline -- no API calls or internet needed after download
- Lightweight -- runs on CPU without issues
- Supports 1,100+ languages with the same architecture
- Simple API -- just tokenize and generate

**Limitations:**
- **Single speaker only** -- no voice cloning or speaker selection
- **Audio quality** -- noticeably lower than Edge TTS or viXTTS; the voice can sound robotic
- **No prosody control** -- cannot adjust speed, pitch, or emotion
- **No SSML support** -- no markup for pauses, emphasis, etc.
- **Long text** -- quality degrades on very long inputs; sentence-level splitting is recommended

### When to use MMS-TTS

| Use case | Recommendation |
|----------|----------------|
| Quick prototyping / offline TTS | MMS-TTS (this notebook) |
| High-quality Vietnamese TTS | Edge TTS or viXTTS |
| Voice cloning | XTTS v2 or viXTTS |
| Low-resource languages | MMS-TTS (best coverage) |

## 8. Key Takeaways

1. **MMS-TTS** provides a quick, lightweight way to generate Vietnamese speech using the VITS architecture.
2. The model is **fully offline** after download -- no API keys, no internet required.
3. It runs comfortably on **CPU** with a model size of ~300 MB.
4. The trade-off is **audio quality**: MMS-TTS sounds more robotic compared to Edge TTS or viXTTS.
5. It is best suited for **prototyping**, **offline applications**, or **low-resource languages** where alternatives do not exist.
6. For production Vietnamese TTS, consider **Edge TTS** (free, high quality, online) or **viXTTS** (high quality, supports voice cloning).